# Phase 11 — Hyperparameter Tuning

**Goal:** tune the Phase 9 selected model using the Phase 10 selected feature group and only the development period (2013–2016).

**Evaluation:** expanding-window walk-forward validation.

**Protected test:** January–February 2017 is never used here.

The search is implemented manually so each candidate is evaluated with a fresh leakage-safe pipeline on every chronological fold.

## 1. Imports and project paths

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "src" / "project_config.py").is_file():
        PROJECT_ROOT = candidate
        break
sys.path.insert(0, str(PROJECT_ROOT))

from src.project_config import *
from src.modeling_utils import *

import json
import numpy as np
import pandas as pd
from sklearn.model_selection import ParameterSampler

## 2. Load the development data

In [2]:
if not FEATURE_DATA.exists():
    raise FileNotFoundError("Run 05_feature_engineering.ipynb first.")

features = pd.read_csv(FEATURE_DATA, parse_dates=["datetime", "target_datetime"])
history_features = (
    [f"PM2.5_lag_{x}h" for x in LAGS]
    + [f"PM2.5_roll_mean_{x}h" for x in ROLL_MEANS]
    + [f"PM2.5_roll_std_{x}h" for x in ROLL_STDS]
)

development = prepare_model_rows(
    features, TARGET, TARGET_TIME, history_features
)
development = development[
    (development[TARGET_TIME] >= pd.Timestamp("2013-03-01 00:00:00"))
    & (development[TARGET_TIME] <= pd.Timestamp("2016-12-31 23:00:00"))
].copy().reset_index(drop=True)

if development.empty:
    raise ValueError("No development rows are available for 2013–2016.")

if not PHASE9_SELECTION.exists():
    raise FileNotFoundError("Run 09_model_comparison.ipynb first.")
SELECTED_MODEL = json.loads(PHASE9_SELECTION.read_text(encoding="utf-8"))["selected_model"]

print("Selected model:", SELECTED_MODEL)
print("Development rows:", f"{len(development):,}")
print("Development target period:", development[TARGET_TIME].min(), "to", development[TARGET_TIME].max())

Selected model: Random Forest
Development rows: 345,348
Development target period: 2013-03-02 01:00:00 to 2016-12-31 23:00:00


## 3. Build the walk-forward folds

In [3]:
def target_range_mask(frame, start, end):
    return (frame[TARGET_TIME] >= pd.Timestamp(start)) & (frame[TARGET_TIME] <= pd.Timestamp(end))

fold_splits = []
for fold in WALK_FORWARD_FOLDS:
    train_mask = target_range_mask(development, fold["train_start"], fold["train_end"])
    valid_mask = target_range_mask(development, fold["validation_start"], fold["validation_end"])
    train_idx = np.flatnonzero(train_mask.to_numpy())
    valid_idx = np.flatnonzero(valid_mask.to_numpy())

    if len(train_idx) == 0 or len(valid_idx) == 0:
        raise ValueError(f"Empty fold detected: {fold['name']}")

    train_last = development.iloc[train_idx][TARGET_TIME].max()
    valid_first = development.iloc[valid_idx][TARGET_TIME].min()

    if train_last >= valid_first:
        raise ValueError(
            f"Temporal leakage detected in {fold['name']}: train ends {train_last}, validation starts {valid_first}."
        )
    if set(train_idx).intersection(valid_idx):
        raise ValueError(f"Row overlap detected in {fold['name']}")

    fold_splits.append((train_idx, valid_idx))

fold_summary = pd.DataFrame([
    {
        "fold": WALK_FORWARD_FOLDS[i]["name"],
        "train_rows": len(train_idx),
        "validation_rows": len(valid_idx),
        "train_target_end": development.iloc[train_idx][TARGET_TIME].max(),
        "validation_target_start": development.iloc[valid_idx][TARGET_TIME].min(),
    }
    for i, (train_idx, valid_idx) in enumerate(fold_splits)
])
display(fold_summary)

,fold,train_rows,validation_rows,train_target_end,validation_target_start
0,fold_1,82652,88188,2013-12-31 23:00:00,2014-01-01
1,fold_2,170840,87353,2014-12-31 23:00:00,2015-01-01
2,fold_3,258193,87155,2015-12-31 23:00:00,2016-01-01


## 4. Select the Phase 10 feature group

In [4]:
DROP_COLUMNS = ["No", "datetime", "target_datetime", "has_next_hour", TARGET]

if not PHASE10_SELECTION.exists():
    raise FileNotFoundError("Run 10_feature_experiments.ipynb and complete all four experiments first.")

feature_payload = json.loads(PHASE10_SELECTION.read_text(encoding="utf-8"))
TUNING_FEATURES = feature_payload["feature_columns"]
missing_features = [c for c in TUNING_FEATURES if c not in development.columns]
if missing_features:
    raise ValueError(f"Phase 10 selected features are missing from the engineered dataset: {missing_features}")

_, NUMERIC_COLUMNS, CATEGORICAL_COLUMNS = model_input_columns(
    development, TARGET, feature_columns=TUNING_FEATURES
)

print("Selected feature group:", feature_payload["selected_feature_group"])
print("Feature count:", len(TUNING_FEATURES))
print("Categorical features:", CATEGORICAL_COLUMNS)

Selected feature group: D_all_including_rolling
Feature count: 33
Categorical features: ['wd', 'station', 'season']


## 5. Define model-specific search spaces

In [5]:
PARAM_GRIDS = {
    "Decision Tree": {
        "model__max_depth": [10, 20, 30],
        "model__min_samples_leaf": [3, 5, 10],
    },
    "Random Forest": {
        "model__n_estimators": [100, 150, 250],
        "model__max_depth": [10, 20, None],
        "model__min_samples_leaf": [2, 3, 5],
    },
    "HistGradientBoosting": {
        "model__max_iter": [150, 200, 300],
        "model__learning_rate": [0.03, 0.05, 0.08],
        "model__max_leaf_nodes": [15, 31, 63],
        "model__l2_regularization": [0.0, 0.1, 1.0],
    },
    "Linear Regression": {},
}

if SELECTED_MODEL not in PARAM_GRIDS:
    raise ValueError(f"Unsupported selected model: {SELECTED_MODEL}")

for parameter_name, values in PARAM_GRIDS[SELECTED_MODEL].items():
    print(parameter_name, "→", values)

model__n_estimators → [100, 150, 250]
model__max_depth → [10, 20, None]
model__min_samples_leaf → [2, 3, 5]


## 6. Search settings

In [6]:
RUN_TUNING = True
N_ITER = 6
RANDOM_STATE = 42

print("RUN_TUNING =", RUN_TUNING)
print("N_ITER =", N_ITER)
print("RANDOM_STATE =", RANDOM_STATE)

RUN_TUNING = True
N_ITER = 6
RANDOM_STATE = 42


## 7. Evaluate one parameter set

In [7]:
def evaluate_parameters(parameter_set):
    """Evaluate one parameter set across all chronological validation folds."""
    fold_records = []

    for fold_number, (train_idx, valid_idx) in enumerate(fold_splits, start=1):
        train = development.iloc[train_idx]
        valid = development.iloc[valid_idx]

        pipeline = make_model_pipeline(SELECTED_MODEL, NUMERIC_COLUMNS, CATEGORICAL_COLUMNS)
        pipeline.set_params(**parameter_set)

        X_train = train[TUNING_FEATURES]
        y_train = train[TARGET]
        X_valid = valid[TUNING_FEATURES]
        y_valid = valid[TARGET]

        # All preprocessing is fitted only on this fold's training rows.
        pipeline.fit(X_train, y_train)
        train_pred = pipeline.predict(X_train)
        valid_pred = pipeline.predict(X_valid)

        train_metrics = regression_metrics(y_train, train_pred)
        valid_metrics = regression_metrics(y_valid, valid_pred)

        fold_records.append({
            "fold": fold_number,
            "train_rows": len(train),
            "validation_rows": len(valid),
            "train_MAE": train_metrics["MAE"],
            "validation_MAE": valid_metrics["MAE"],
            "train_RMSE": train_metrics["RMSE"],
            "validation_RMSE": valid_metrics["RMSE"],
            "train_R2": train_metrics["R2"],
            "validation_R2": valid_metrics["R2"],
        })

    fold_results = pd.DataFrame(fold_records)
    summary = {
        "mean_validation_MAE": fold_results["validation_MAE"].mean(),
        "mean_validation_RMSE": fold_results["validation_RMSE"].mean(),
        "mean_validation_R2": fold_results["validation_R2"].mean(),
        "validation_RMSE_std": fold_results["validation_RMSE"].std(ddof=0),
        "mean_train_RMSE": fold_results["train_RMSE"].mean(),
    }
    summary["RMSE_train_to_validation_gap"] = summary["mean_validation_RMSE"] - summary["mean_train_RMSE"]
    return summary, fold_results

## 8. Run the randomized search

In [8]:
RESULT_DIR.mkdir(parents=True, exist_ok=True)
tuning_result = None
best_params = {}

if SELECTED_MODEL == "Linear Regression":
    print("Linear Regression has no meaningful hyperparameter grid in this project.")

elif RUN_TUNING:
    sampled_parameters = list(
        ParameterSampler(
            PARAM_GRIDS[SELECTED_MODEL],
            n_iter=N_ITER,
            random_state=RANDOM_STATE,
        )
    )

    records = []
    for candidate_id, parameter_set in enumerate(sampled_parameters, start=1):
        print(f"\nEvaluating candidate {candidate_id}/{len(sampled_parameters)}")
        print(parameter_set)

        summary_metrics, _ = evaluate_parameters(parameter_set)
        row = {
            "candidate_id": candidate_id,
            "model": SELECTED_MODEL,
            **parameter_set,
            **summary_metrics,
        }
        records.append(row)
        pd.DataFrame(records).to_csv(PHASE11_RESULTS, index=False)

    tuning_result = (
        pd.DataFrame(records)
        .sort_values(["mean_validation_RMSE", "mean_validation_MAE", "validation_RMSE_std"])
        .reset_index(drop=True)
    )

    best_row = tuning_result.iloc[0]
    best_params = {
        parameter_name: (
            best_row[parameter_name].item()
            if isinstance(best_row[parameter_name], np.generic)
            else best_row[parameter_name]
        )
        for parameter_name in PARAM_GRIDS[SELECTED_MODEL]
    }

    PHASE11_BEST_PARAMS.write_text(
        json.dumps({
            "selected_model": SELECTED_MODEL,
            "best_params": best_params,
            "scoring": "mean validation RMSE",
            "cv": "expanding-window walk-forward",
            "test_set_used": False,
            "n_iter": len(sampled_parameters),
            "random_state": RANDOM_STATE,
        }, indent=2),
        encoding="utf-8",
    )

    display(tuning_result.round(4))
    print("Best parameters:", best_params)
    print("Created:", PHASE11_RESULTS)
    print("Created:", PHASE11_BEST_PARAMS)
else:
    print("Tuning disabled. Set RUN_TUNING = True to execute the search.")


Evaluating candidate 1/6
{'model__n_estimators': 250, 'model__min_samples_leaf': 5, 'model__max_depth': 10}

Evaluating candidate 2/6
{'model__n_estimators': 150, 'model__min_samples_leaf': 3, 'model__max_depth': 20}

Evaluating candidate 3/6
{'model__n_estimators': 100, 'model__min_samples_leaf': 2, 'model__max_depth': 20}

Evaluating candidate 4/6
{'model__n_estimators': 100, 'model__min_samples_leaf': 3, 'model__max_depth': None}

Evaluating candidate 5/6
{'model__n_estimators': 100, 'model__min_samples_leaf': 2, 'model__max_depth': 10}

Evaluating candidate 6/6
{'model__n_estimators': 250, 'model__min_samples_leaf': 2, 'model__max_depth': 20}


,candidate_id,model,model__n_estimators,model__min_samples_leaf,model__max_depth,mean_validation_MAE,mean_validation_RMSE,mean_validation_R2,validation_RMSE_std,mean_train_RMSE,RMSE_train_to_validation_gap
0,6,Random Forest,250,2,20.0,10.7514,22.3973,0.9210,2.5666,10.7077,11.6896
1,2,Random Forest,150,3,20.0,10.7426,22.3991,0.9210,2.5530,11.5409,10.8582
2,4,Random Forest,100,3,NaN,10.7891,22.4601,0.9206,2.6123,10.0136,12.4465
3,3,Random Forest,100,2,20.0,10.7885,22.4685,0.9205,2.5907,10.7493,11.7192
4,1,Random Forest,250,5,10.0,10.8047,22.5331,0.9201,2.3772,17.4864,5.0467
5,5,Random Forest,100,2,10.0,10.8317,22.5445,0.9200,2.4334,17.1744,5.3701


Best parameters: {'model__n_estimators': 250, 'model__max_depth': 20.0, 'model__min_samples_leaf': 2}
Created: /Users/ruchi/Downloads/phase12_fixed/data/results/phase11_tuning_results.csv
Created: /Users/ruchi/Downloads/phase12_fixed/data/results/phase11_best_parameters.json


## 9. Final leakage check

In [10]:
final_test_path = SPLIT_DIR / "final_test_2017_01_to_02.csv"
if not final_test_path.exists():
    raise FileNotFoundError("Protected 2017 test file not found. Run 06_data_split.ipynb first.")

final_test = pd.read_csv(final_test_path, parse_dates=[TARGET_TIME])
test_start = final_test[TARGET_TIME].min()
if development[TARGET_TIME].max() >= test_start:
    raise ValueError("Leakage check failed: development data overlaps the final test period.")

print("Final holdout is protected:", test_start, "to", final_test[TARGET_TIME].max())
print("Hyperparameter tuning stage complete.")

Final holdout is protected: 2017-01-01 00:00:00 to 2017-02-28 23:00:00
Hyperparameter tuning stage complete.
